# Feature engineering and machine-learning results

Played matches only; scheduled fixtures are 'Coming soon' and never analysed as results. Source: PostgreSQL warehouse (`docker compose up -d`) and saved reports.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)


## Leakage-free features
Every feature of a match uses only matches from strictly earlier dates. `tests/test_ml.py` proves it by changing all later results and checking nothing earlier moves.

In [2]:
from fbplatform.ml import features as F
df = F.build_match_features()
tr, va, te = F.chronological_split(df)
pd.DataFrame({'set': ['train', 'validation', 'test'], 'matches': [len(tr), len(va), len(te)], 'from': [x.match_date.min().date() for x in (tr, va, te)], 'to': [x.match_date.max().date() for x in (tr, va, te)]})

,set,matches,from,to
0,train,1242,2000-06-10,2022-11-23
1,validation,262,2022-11-24,2024-11-15
2,test,275,2024-11-16,2026-10-06


In [3]:
df[F.MATCH_FEATS].describe().T[['count', 'mean', 'std', 'min', 'max']].round(3).head(30)

,count,mean,std,min,max
a_n_prior,1779.0,27.714,22.938,0.000,105.000
a_win_pct,1729.0,0.397,0.200,0.000,1.000
a_ppm,1729.0,1.428,0.557,0.000,3.000
a_gf_avg,1729.0,1.266,0.475,0.000,4.000
a_ga_avg,1729.0,1.196,0.486,0.000,5.000
a_gd_avg,1729.0,0.070,0.820,-5.000,3.000
a_cs_pct,1729.0,0.332,0.173,0.000,1.000
a_form_ppm,1729.0,1.442,0.725,0.000,3.000
a_form_gd,1729.0,0.091,1.042,-5.000,3.200
a_editions,1779.0,6.236,4.543,0.000,19.000


## Match outcome models (test period)
The one-feature Elo baseline is the bar to beat.

In [4]:
c = pd.read_csv(ROOT / 'reports/ml/outcome/match_outcome_model_comparison.csv')
c[c.split == 'test'][['model', 'accuracy', 'accuracy_ci_low', 'accuracy_ci_high', 'f1_macro', 'log_loss', 'log_loss_ci_low', 'log_loss_ci_high', 'roc_auc_ovr']].round(3)

,model,accuracy,accuracy_ci_low,accuracy_ci_high,f1_macro,log_loss,log_loss_ci_low,log_loss_ci_high,roc_auc_ovr
1,baseline_prior,0.433,0.375,0.491,0.201,1.075,1.051,1.101,0.500
3,baseline_elo_logreg,0.567,0.509,0.622,0.430,0.942,0.897,0.988,0.711
5,logreg,0.542,0.484,0.600,0.442,0.951,0.893,1.013,0.675
7,random_forest,0.556,0.498,0.611,0.421,0.972,0.931,1.016,0.685
9,xgboost,0.575,0.520,0.629,0.443,0.956,0.910,1.002,0.695


In [5]:
pd.read_csv(ROOT / 'reports/ml/outcome/permutation_importance.csv').head(10).round(4)

,feature,importance_mean,importance_std
0,editions_diff,0.0578,0.0094
1,elo_diff,0.0114,0.0025
2,gd_avg_diff,0.0107,0.0036
3,att_strength_diff,0.0039,0.0017
4,form_gd_diff,0.0038,0.0014
5,def_strength_diff,0.0019,0.0012
6,b_form_gd,0.0016,0.0017
7,a_ga_avg,0.0014,0.0020
8,a_elo,0.0014,0.0006
9,win_pct_diff,0.0011,0.0020


In [6]:
pd.read_csv(ROOT / 'reports/ml/outcome/shap_mean_abs.csv').head(10).round(3)

,feature,A win,draw,B win,mean_abs_shap
0,editions_diff,0.149,0.005,0.218,0.124
1,gd_avg_diff,0.101,0.029,0.033,0.054
2,elo_diff,0.041,0.001,0.076,0.039
3,form_gd_diff,0.013,0.008,0.045,0.022
4,att_strength_diff,0.007,0.004,0.055,0.022
5,win_pct_diff,0.048,0.012,0.005,0.022
6,a_ga_avg,0.005,0.043,0.004,0.017
7,def_strength_diff,0.032,0.005,0.015,0.017
8,b_form_gd,0.006,0.026,0.014,0.015
9,b_n_prior,0.031,0.010,0.000,0.014


## Goal models

In [7]:
pd.read_csv(ROOT / 'reports/ml/goals/goal_model_comparison.csv').round(3)

,split,n_rows,best_params,model,mae_team_goals,rmse_team_goals,poisson_deviance,mae_total_goals,rmse_total_goals,cv_poisson_deviance_train
0,validation,512,NaN,baseline_train_mean,0.957,1.192,1.297,1.297,1.579,NaN
1,test,548,NaN,baseline_train_mean,1.034,1.371,1.451,1.416,1.816,NaN
2,validation,512,{'m__alpha': 1.0},poisson_regression,0.880,1.117,1.160,1.287,1.577,1.215
3,test,548,{'m__alpha': 1.0},poisson_regression,0.935,1.255,1.230,1.388,1.780,1.215
4,validation,512,"{'m__max_depth': 3, 'm__min_samples_leaf': 15}",random_forest,0.889,1.123,1.173,1.286,1.575,1.235
5,test,548,"{'m__max_depth': 3, 'm__min_samples_leaf': 15}",random_forest,0.957,1.272,1.261,1.402,1.800,1.235
6,validation,512,"{'learning_rate': 0.03, 'max_depth': 2, 'min_child_weigh...",xgboost,0.888,1.124,1.174,1.290,1.580,1.226
7,test,548,"{'learning_rate': 0.03, 'max_depth': 2, 'min_child_weigh...",xgboost,0.955,1.268,1.257,1.405,1.797,1.226


In [8]:
pd.read_csv(ROOT / 'reports/ml/goals/count_model_diagnostics.csv').round(3)

,pearson_dispersion,zero_share_observed,zero_share_poisson_expected,model
0,1.144,0.301,0.293,poisson_regression


## Clustering and player analysis

In [9]:
pd.read_csv(ROOT / 'reports/ml/clustering/team_cluster_profiles.csv')[['n_teams', 'gf_pm', 'ga_pm', 'win_pct', 'suggested_description', 'members']]

,n_teams,gf_pm,ga_pm,win_pct,suggested_description,members
0,45,0.977,1.506,25.553,weak at both ends,"Croatia, Czech Republic, Poland, Paraguay, Ecuador, Peru..."
1,32,1.413,1.033,46.153,dominant: strong attack and strong defence,"France, Spain, Germany, Portugal, England, Netherlands, ..."


In [10]:
pd.read_csv(ROOT / 'reports/ml/clustering/player_cluster_profiles.csv')[['position', 'cluster', 'n_players', 'description', 'examples']]

,position,cluster,n_players,description,examples
0,Defender,0,78,"Defender: key passing, chance-creating, assisting","Ricardo Iván Rodríguez Araya, Joshua Kimmich, Harry Magu..."
1,Defender,1,260,Defender: average profile,"John Stones, Kyle Walker, Jan Vertonghen, Manuel Obafemi..."
2,Midfielder,0,158,Midfielder: average profile,"Granit Xhaka, Declan Rice, Axel Witsel, Sergio Busquets ..."
3,Midfielder,1,67,"Midfielder: shooting, key passing, chance-taking","Antoine Griezmann, Luka Modrić, Kevin De Bruyne, Toni Kr..."
4,Forward,0,41,"Forward: chance-creating, assisting, key passing","Ivan Perišić, Kylian Mbappé Lottin, Lionel Andrés Messi ..."
5,Forward,1,71,Forward: average profile,"Bernardo Mota Veiga de Carvalho e Silva, Andrej Kramarić..."
6,Forward,2,66,"Forward: chance-taking, scoring","Harry Kane, Cristiano Ronaldo dos Santos Aveiro, Romelu ..."


In [11]:
pd.read_csv(ROOT / 'reports/ml/players/next_tournament_prediction.csv').round(3)

,model,n_train,n_test,mae,r2,spearman,note
0,baseline_train_mean,427,811,0.163,-0.001,NaN,"chronological split: train editions <= 2016, test later;..."
1,baseline_last_edition_rate,427,811,0.137,-0.267,0.350,"chronological split: train editions <= 2016, test later;..."
2,ridge,427,811,0.133,0.269,0.459,"chronological split: train editions <= 2016, test later;..."
3,random_forest,427,811,0.125,0.263,0.468,"chronological split: train editions <= 2016, test later;..."
